# 5 · QPROP Input Generator

This notebook builds one QPROP propeller input file per sampled propeller configuration.

The notebook is intentionally narrow in scope: it does not generate geometries, does not run XFOIL, and does not run QPROP. Its only responsibility is to combine:

- the geometric propeller parameters,
- the aerodynamic coefficients extracted from XFOIL,
- the selected aerodynamic root station,

and write clean `.txt` propeller files that can be passed to QPROP.

The root station can be selected with a single boolean constant:

- `USE_HUB_PROFILE_FOR_QPROP = True` uses the aerodynamic hub station at `8.25 mm`.
- `USE_HUB_PROFILE_FOR_QPROP = False` uses the inner construction station at `4.5 mm`.

In both cases, each propeller file contains exactly three radial stations: root, mid, and outer.


---

## 1. Imports

The notebook uses standard data-processing tools and `CubicSpline` for the hub-station geometry interpolation.


In [1]:
# Imports

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from tqdm.auto import tqdm


c:\Users\hecto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 2. Pipeline constants

All settings that may need to change later are defined here.

The most important switch is `USE_HUB_PROFILE_FOR_QPROP`:

- `True`: the first aerodynamic station is the hub station at the physical outer hub radius.
- `False`: the first aerodynamic station is the inner profile station.

This keeps the notebook flexible if the CAD/aerodynamic interface changes later.


In [2]:
# -----------------------------
# Input / output paths
# -----------------------------

BASE_DIR = Path.cwd()
CSV_DIR = BASE_DIR / "csv"

GEOMETRY_CSV_PATH = CSV_DIR / "prop_geometrical_params.csv"
XFOIL_CSV_PATH = CSV_DIR / "xfoil_profile_simulation.csv"

QPROP_INPUT_DIR = BASE_DIR / "qprop_input"
QPROP_INPUT_EXTENSION = ".txt"


# -----------------------------
# Root-station selection
# -----------------------------

# True  -> QPROP root station is the hub station at HUB_OUTER_RADIUS_MM.
# False -> QPROP root station is the inner station at INNER_PROFILE_RADIUS_MM.
USE_HUB_PROFILE_FOR_QPROP = True

ROOT_STATION = "hub" if USE_HUB_PROFILE_FOR_QPROP else "inner"
QPROP_STATIONS = [ROOT_STATION, "mid", "outer"]


# -----------------------------
# Radial station constants [mm]
# -----------------------------

# Inner construction profile radius.
INNER_PROFILE_RADIUS_MM = 4.5

# Physical outer hub radius. If hub mode is selected, QPROP starts here.
HUB_OUTER_RADIUS_MM = 8.25

# The hub chord and hub twist angle are obtained by a cubic spline through
# the inner, mid, and outer geometry control sections.
SPLINE_BC_TYPE = "natural"
CLAMP_INTERPOLATED_GEOMETRY = True


# -----------------------------
# QPROP format constants
# -----------------------------

# The station table is written in millimetres. QPROP then applies these
# factors to convert radius and chord to metres.
QPROP_RADIUS_FACTOR = 0.001
QPROP_CHORD_FACTOR = 0.001
QPROP_BETA_FACTOR = 1.0

QPROP_RADIUS_ADD = 0.0
QPROP_CHORD_ADD = 0.0
QPROP_BETA_ADD = 0.0

# Reynolds exponent used by the drag polar model.
RE_EXPONENT = -0.5


# -----------------------------
# Aerodynamic coefficient columns required by QPROP
# -----------------------------

AERO_KEYS = [
    "CL0", "CL_a", "CLmin", "CLmax", "CD0", "CD2u", "CD2l",
    "CLCD0", "REref", "REexp",
]


# -----------------------------
# Output control
# -----------------------------

OVERWRITE_EXISTING_FILES = True
CONFIG_ID_DIGITS = 4

print(f"Root station selected : {ROOT_STATION}")
print(f"QPROP stations        : {QPROP_STATIONS}")
print(f"Root radius [mm]      : {HUB_OUTER_RADIUS_MM if USE_HUB_PROFILE_FOR_QPROP else INNER_PROFILE_RADIUS_MM}")


Root station selected : hub
QPROP stations        : ['hub', 'mid', 'outer']
Root radius [mm]      : 8.25


---

## 3. Load input tables

The geometry table provides the radial station positions, chord lengths, blade angles, and blade count.

The XFOIL table provides the aerodynamic polar coefficients for the selected station set. Therefore:

- if `USE_HUB_PROFILE_FOR_QPROP = True`, the XFOIL CSV must contain `hub`, `mid`, and `outer` coefficients;
- if `USE_HUB_PROFILE_FOR_QPROP = False`, the XFOIL CSV must contain `inner`, `mid`, and `outer` coefficients.

NACA codes are read as strings so that leading zeros are never lost.


In [3]:
# Load input CSV files

if not GEOMETRY_CSV_PATH.exists():
    raise FileNotFoundError(f"Geometry CSV not found: {GEOMETRY_CSV_PATH}")

if not XFOIL_CSV_PATH.exists():
    raise FileNotFoundError(
        f"XFOIL profile simulation CSV not found: {XFOIL_CSV_PATH}. "
        "Run the XFOIL profile simulation step before generating QPROP files."
    )

geometry_df_raw = pd.read_csv(GEOMETRY_CSV_PATH)

naca_dtype = {"naca_hub": str, "naca_inner": str, "naca_mid": str, "naca_outer": str}
xfoil_df = pd.read_csv(XFOIL_CSV_PATH, dtype=naca_dtype)

GEOMETRY_RENAME_MAP = {
    "ring height": "ring_height",
    "ring thickness": "ring_thickness",
    "blade count": "blade_count",
    "inner thickness": "inner_thickness",
    "inner max pos": "inner_max_pos",
    "inner camber": "inner_camber",
    "inner chord": "inner_chord",
    "inner angle": "inner_angle",
    "mid radial pos": "mid_radial_pos",
    "mid chord": "mid_chord",
    "mid angle": "mid_angle",
    "outer thickness": "outer_thickness",
    "outer max pos": "outer_max_pos",
    "outer camber": "outer_camber",
    "outer chord": "outer_chord",
    "outer angle": "outer_angle",
}

geometry_df = geometry_df_raw.rename(columns=GEOMETRY_RENAME_MAP)

required_geometry_columns = [
    "config_id", "radius", "blade_count", "inner_chord", "inner_angle",
    "mid_radial_pos", "mid_chord", "mid_angle", "outer_chord", "outer_angle",
]

missing_geometry = [col for col in required_geometry_columns if col not in geometry_df.columns]
if missing_geometry:
    raise ValueError(f"Missing required geometry columns: {missing_geometry}")

required_xfoil_columns = ["config_id"]
for station in QPROP_STATIONS:
    required_xfoil_columns.extend([f"naca_{station}", f"re_{station}", f"tier_{station}"])
    required_xfoil_columns.extend([f"{station}_{key}" for key in AERO_KEYS])

missing_xfoil = [col for col in required_xfoil_columns if col not in xfoil_df.columns]
if missing_xfoil:
    raise ValueError(
        "The XFOIL CSV does not match the selected root-station mode. "
        f"Missing columns: {missing_xfoil}"
    )

for station in QPROP_STATIONS:
    xfoil_df[f"naca_{station}"] = xfoil_df[f"naca_{station}"].astype(str).str.zfill(4)

print(f"Geometry rows : {len(geometry_df)}")
print(f"XFOIL rows    : {len(xfoil_df)}")
print(f"Geometry path : {GEOMETRY_CSV_PATH}")
print(f"XFOIL path    : {XFOIL_CSV_PATH}")


Geometry rows : 5000
XFOIL rows    : 5000
Geometry path : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\prop_geometrical_params.csv
XFOIL path    : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\xfoil_profile_simulation.csv


---

## 4. Merge geometry and aerodynamic data

The merge is performed on `config_id`. This guarantees that the aerodynamic coefficients used in each QPROP file correspond to the same propeller geometry.


In [4]:
# Merge input tables

merged_df = pd.merge(
    geometry_df,
    xfoil_df,
    on="config_id",
    how="inner",
    validate="one_to_one",
)

merged_df = merged_df.sort_values("config_id").reset_index(drop=True)

if len(merged_df) != len(geometry_df):
    raise ValueError(
        "The merged table does not contain one row per geometry configuration. "
        "Check that config_id values match between the geometry and XFOIL CSV files."
    )

print(f"Merged rows: {len(merged_df)}")
merged_df.head()


Merged rows: 5000


,config_id,radius,ring_height,ring_thickness,blade_count,inner_thickness,inner_max_pos,inner_camber,inner_chord,inner_angle,...,outer_CL_a,outer_CLmin,outer_CLmax,outer_CD0,outer_CD2u,outer_CD2l,outer_CLCD0,outer_REref,outer_REexp,outer_xfoil_ok
0,0,67,5,2,5,17,6,4,7,23,...,4.0000,-0.4253,0.8684,0.035460,0.251010,0.251010,0.1060,20000,-0.5,True
1,1,60,4,1,3,19,5,6,11,25,...,6.1848,-0.3958,0.9688,0.075877,0.102049,0.102049,-0.0473,25000,-0.5,True
2,2,80,10,4,5,18,5,5,6,17,...,8.1089,-0.4415,1.0536,0.023710,0.010000,0.010000,-0.0249,40000,-0.5,True
3,3,66,9,1,6,22,4,7,5,22,...,6.9417,-0.3646,1.9225,0.008000,0.020000,0.020000,0.2262,20000,-0.5,False
4,4,67,9,2,5,18,4,8,6,22,...,4.2840,-0.5403,0.6678,0.040446,0.124572,0.124572,-0.3460,30000,-0.5,True


---

## 5. Helper functions

The helper functions below handle three things:

1. radial station geometry,
2. aerodynamic coefficient extraction,
3. conversion into the station format required by QPROP.

When hub mode is active, the hub chord and hub blade angle are evaluated using a cubic spline through the inner, mid, and outer geometry stations. The aerodynamic coefficients are not interpolated in this notebook. They are read directly from the XFOIL results for the hub profile.


In [5]:
# Helper functions


def weighted_average(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    if np.any(weights <= 0):
        raise ValueError("All weights must be positive for chord-weighted averaging.")
    return float(np.sum(values * weights) / np.sum(weights))


def cubic_spline_value(x_points, y_points, x_eval):
    x = np.asarray(x_points, dtype=float)
    y = np.asarray(y_points, dtype=float)
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    if len(np.unique(x)) != len(x):
        raise ValueError(f"Spline control radii must be unique. Received: {x.tolist()}")
    spline = CubicSpline(x, y, bc_type=SPLINE_BC_TYPE)
    return float(spline(float(x_eval)))


def clamp_to_source_range(value, source_values):
    source_values = np.asarray(source_values, dtype=float)
    return float(np.clip(value, source_values.min(), source_values.max()))


def source_radii_mm(row):
    tip_radius_mm = float(row["radius"])
    return {
        "inner": float(INNER_PROFILE_RADIUS_MM),
        "mid": float(row["mid_radial_pos"]) * tip_radius_mm,
        "outer": tip_radius_mm,
    }


def source_geometry(row):
    radii = source_radii_mm(row)
    return {
        "inner": {"name": "inner", "radius_mm": radii["inner"], "chord_mm": float(row["inner_chord"]), "beta_deg": float(row["inner_angle"])},
        "mid": {"name": "mid", "radius_mm": radii["mid"], "chord_mm": float(row["mid_chord"]), "beta_deg": float(row["mid_angle"])},
        "outer": {"name": "outer", "radius_mm": radii["outer"], "chord_mm": float(row["outer_chord"]), "beta_deg": float(row["outer_angle"])},
    }


def hub_geometry(row):
    geom = source_geometry(row)
    x_points = [geom[station]["radius_mm"] for station in ["inner", "mid", "outer"]]
    chord_values = [geom[station]["chord_mm"] for station in ["inner", "mid", "outer"]]
    beta_values = [geom[station]["beta_deg"] for station in ["inner", "mid", "outer"]]

    chord_mm = cubic_spline_value(x_points, chord_values, HUB_OUTER_RADIUS_MM)
    beta_deg = cubic_spline_value(x_points, beta_values, HUB_OUTER_RADIUS_MM)

    if CLAMP_INTERPOLATED_GEOMETRY:
        chord_mm = clamp_to_source_range(chord_mm, chord_values)
        beta_deg = clamp_to_source_range(beta_deg, beta_values)

    return {"name": "hub", "radius_mm": float(HUB_OUTER_RADIUS_MM), "chord_mm": float(chord_mm), "beta_deg": float(beta_deg)}


def station_aero(row, station):
    aero = {}
    for key in AERO_KEYS:
        value = row[f"{station}_{key}"]
        if key == "REref":
            aero[key] = int(round(float(value)))
        elif key == "REexp":
            aero[key] = float(RE_EXPONENT)
        else:
            aero[key] = float(value)
    return aero


def station_metadata(row, station):
    return {
        "naca_code": str(row[f"naca_{station}"]).zfill(4),
        "reynolds": int(round(float(row[f"re_{station}"]))),
        "tier": str(row[f"tier_{station}"]),
    }


def build_qprop_stations(row):
    geom = source_geometry(row)
    if USE_HUB_PROFILE_FOR_QPROP:
        root_geom = hub_geometry(row)
        root_station_name = "hub"
    else:
        root_geom = geom["inner"]
        root_station_name = "inner"

    stations = []
    for station_name, station_geom in [(root_station_name, root_geom), ("mid", geom["mid"]), ("outer", geom["outer"])]:
        metadata = station_metadata(row, station_name)
        stations.append({**station_geom, **metadata, "aero": station_aero(row, station_name)})

    return sorted(stations, key=lambda item: item["radius_mm"])


def build_global_defaults(stations):
    chords = [station["chord_mm"] for station in stations]
    defaults = {}
    for key in AERO_KEYS:
        values = [station["aero"][key] for station in stations]
        if key == "REref":
            defaults[key] = int(round(weighted_average(values, chords)))
        elif key == "REexp":
            defaults[key] = float(RE_EXPONENT)
        else:
            defaults[key] = float(weighted_average(values, chords))
    return defaults


---

## 6. QPROP file writer

Each output file contains:

1. propeller name,
2. blade count, tip radius, and selected root radius,
3. global aerodynamic defaults,
4. QPROP conversion factors,
5. three station rows: root, mid, and outer.

The station rows include the local aerodynamic coefficients extracted from the XFOIL table. A short comment is added to each row with the station name, NACA code, Reynolds number, and polar tier for traceability.


In [6]:
# QPROP input writer


def selected_root_radius_mm():
    return HUB_OUTER_RADIUS_MM if USE_HUB_PROFILE_FOR_QPROP else INNER_PROFILE_RADIUS_MM


def format_station_line(station):
    aero = station["aero"]
    return (
        f'{station["radius_mm"]:7.2f}  '
        f'{station["chord_mm"]:9.2f}  '
        f'{station["beta_deg"]:9.2f}  '
        f'{aero["CL0"]:7.4f}  '
        f'{aero["CL_a"]:7.4f}  '
        f'{aero["CLmin"]:7.4f}  '
        f'{aero["CLmax"]:7.4f}  '
        f'{aero["CD0"]:8.5f}  '
        f'{aero["CD2u"]:8.5f}  '
        f'{aero["CD2l"]:8.5f}  '
        f'{aero["CLCD0"]:7.4f}  '
        f'{int(aero["REref"]):6d}  '
        f'{aero["REexp"]:.3f}'
        f'   ! {station["name"]}  NACA {station["naca_code"]}  Re {station["reynolds"]}  tier {station["tier"]}'
    )


def build_qprop_file_text(row):
    config_id = int(row["config_id"])
    blade_count = int(row["blade_count"])
    radius_tip_m = float(row["radius"]) / 1000.0
    radius_root_m = selected_root_radius_mm() / 1000.0

    stations = build_qprop_stations(row)
    defaults = build_global_defaults(stations)

    lines = [
        f"Propeller_{config_id:0{CONFIG_ID_DIGITS}d}",
        f"{blade_count}  {radius_tip_m:.5f}  {radius_root_m:.5f}   ! Nblades  R_tip(m)  R_root(m)",
        f"! Root station mode: {ROOT_STATION}",
        "",
        "! --- Global defaults: chord-weighted average of the three station data ---",
        f'{defaults["CL0"]:.4f}  {defaults["CL_a"]:.4f}   ! CL0  CL_a',
        f'{defaults["CLmin"]:.4f}  {defaults["CLmax"]:.4f}   ! CLmin  CLmax',
        f'{defaults["CD0"]:.5f}  {defaults["CD2u"]:.5f}  {defaults["CD2l"]:.5f}  {defaults["CLCD0"]:.4f}   ! CD0  CD2u  CD2l  CLCD0',
        f'{int(defaults["REref"])}  {defaults["REexp"]:.3f}   ! REref  REexp',
        "",
        f"{QPROP_RADIUS_FACTOR:.3f}  {QPROP_CHORD_FACTOR:.3f}  {QPROP_BETA_FACTOR:.1f}   ! Rfac Cfac Bfac",
        f"{QPROP_RADIUS_ADD:.1f}    {QPROP_CHORD_ADD:.1f}    {QPROP_BETA_ADD:.1f}   ! Radd Cadd Badd",
        "",
        "! r(mm) chord(mm) beta(deg) CL0 CL_a CLmin CLmax CD0 CD2u CD2l CLCD0 REref REexp",
    ]

    for station in stations:
        lines.append(format_station_line(station))

    lines.append("")
    return "\n".join(lines) + "\n"


def write_qprop_input_file(row):
    config_id = int(row["config_id"])
    output_path = QPROP_INPUT_DIR / f"prop_{config_id:0{CONFIG_ID_DIGITS}d}{QPROP_INPUT_EXTENSION}"
    if output_path.exists() and not OVERWRITE_EXISTING_FILES:
        return output_path
    output_path.write_text(build_qprop_file_text(row), encoding="utf-8")
    return output_path


---

## 7. Generate all QPROP input files

The output folder is created automatically. If `OVERWRITE_EXISTING_FILES = True`, any previous files in the folder are removed before generating the new set.


In [7]:
# Generate QPROP input files

if OVERWRITE_EXISTING_FILES and QPROP_INPUT_DIR.exists():
    shutil.rmtree(QPROP_INPUT_DIR)

QPROP_INPUT_DIR.mkdir(parents=True, exist_ok=True)

output_files = []
for _, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Writing QPROP input files"):
    output_files.append(write_qprop_input_file(row))

print(f"Generated files : {len(output_files)}")
print(f"Output folder   : {QPROP_INPUT_DIR}")
if output_files:
    print(f"First file      : {output_files[0]}")
    print(f"Last file       : {output_files[-1]}")


Writing QPROP input files: 100%|██████████| 5000/5000 [00:07<00:00, 686.97it/s]

Generated files : 5000
Output folder   : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_input
First file      : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_input\prop_0000.txt
Last file       : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_input\prop_4999.txt


---

## 8. Preview one generated file

The preview is useful for checking the file structure before passing the files to QPROP.


In [8]:
# Preview the first generated QPROP input file

if not output_files:
    raise RuntimeError("No QPROP input files were generated.")

preview_path = output_files[0]
print(preview_path)
print("-" * 72)
print(preview_path.read_text(encoding="utf-8")[:2000])


c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_input\prop_0000.txt
------------------------------------------------------------------------
Propeller_0000
5  0.06700  0.00825   ! Nblades  R_tip(m)  R_root(m)
! Root station mode: hub

! --- Global defaults: chord-weighted average of the three station data ---
-0.0593  3.8690   ! CL0  CL_a
-0.3675  0.7583   ! CLmin  CLmax
0.04538  0.22501  0.22501  -0.0364   ! CD0  CD2u  CD2l  CLCD0
17791  -0.500   ! REref  REexp

0.001  0.001  1.0   ! Rfac Cfac Bfac
0.0    0.0    0.0   ! Radd Cadd Badd

! r(mm) chord(mm) beta(deg) CL0 CL_a CLmin CLmax CD0 CD2u CD2l CLCD0 REref REexp
   8.25       9.64      20.93  -0.1565   3.6289  -0.1378   0.5968   0.05989   0.20856   0.20856  -0.1372   10000  -0.500   ! hub  NACA 4617  Re 10000  tier viscous
  33.50      22.00       9.00  -0.0847   3.9028  -0.4366   0.7690   0.04443   0.21804   0.21804  -0.0699   20000  -0.500   ! mid  NACA 4615  Re 20000  tier viscous
  67.00      12.00       3.

---

## 9. Final checks

The final checks verify that:

- one file was generated per configuration;
- each file contains three radial stations;
- the first station is located at the selected root radius;
- station radii are strictly increasing;
- the selected aerodynamic columns exist and were used.


In [9]:
# Final consistency checks

all_ok = True

def check(condition, message):
    global all_ok
    status = "OK" if bool(condition) else "FAIL"
    print(f"[{status:4s}] {message}")
    if not condition:
        all_ok = False

check(len(output_files) == len(merged_df), "one QPROP input file per configuration")
check(all(path.exists() and path.stat().st_size > 0 for path in output_files), "all output files are non-empty")

expected_station_count = 3
expected_root_radius = selected_root_radius_mm()
station_counts = []
first_station_radii = []
radii_are_increasing = []
root_station_names = []

for _, row in merged_df.iterrows():
    stations = build_qprop_stations(row)
    radii = [station["radius_mm"] for station in stations]
    station_counts.append(len(stations))
    first_station_radii.append(radii[0])
    radii_are_increasing.append(all(np.diff(radii) > 0))
    root_station_names.append(stations[0]["name"])

check(all(count == expected_station_count for count in station_counts), "each QPROP file uses exactly three radial stations")
check(all(radii_are_increasing), "station radii are strictly increasing")
check(np.allclose(first_station_radii, expected_root_radius), f"first station radius is {expected_root_radius} mm for the selected mode")
check(set(root_station_names) == {ROOT_STATION}, f"first station name is always '{ROOT_STATION}'")

for station in QPROP_STATIONS:
    check(f"naca_{station}" in merged_df.columns, f"naca_{station} column exists")
    check(f"re_{station}" in merged_df.columns, f"re_{station} column exists")
    for key in AERO_KEYS:
        check(f"{station}_{key}" in merged_df.columns, f"{station}_{key} column exists")

sample_text = output_files[0].read_text(encoding="utf-8") if output_files else ""
check(f"Root station mode: {ROOT_STATION}" in sample_text, "generated file records the selected root-station mode")
check(f"{expected_root_radius:7.2f}" in sample_text, "generated file contains the selected root radius")

print()
print("Selected QPROP root mode")
print("-" * 72)
print(f"USE_HUB_PROFILE_FOR_QPROP = {USE_HUB_PROFILE_FOR_QPROP}")
print(f"ROOT_STATION              = {ROOT_STATION}")
print(f"ROOT_RADIUS_MM            = {expected_root_radius}")
print(f"QPROP_STATIONS            = {QPROP_STATIONS}")
print()
print("=" * 72)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED")
print("=" * 72)


[OK  ] one QPROP input file per configuration
[OK  ] all output files are non-empty
[OK  ] each QPROP file uses exactly three radial stations
[OK  ] station radii are strictly increasing
[OK  ] first station radius is 8.25 mm for the selected mode
[OK  ] first station name is always 'hub'
[OK  ] naca_hub column exists
[OK  ] re_hub column exists
[OK  ] hub_CL0 column exists
[OK  ] hub_CL_a column exists
[OK  ] hub_CLmin column exists
[OK  ] hub_CLmax column exists
[OK  ] hub_CD0 column exists
[OK  ] hub_CD2u column exists
[OK  ] hub_CD2l column exists
[OK  ] hub_CLCD0 column exists
[OK  ] hub_REref column exists
[OK  ] hub_REexp column exists
[OK  ] naca_mid column exists
[OK  ] re_mid column exists
[OK  ] mid_CL0 column exists
[OK  ] mid_CL_a column exists
[OK  ] mid_CLmin column exists
[OK  ] mid_CLmax column exists
[OK  ] mid_CD0 column exists
[OK  ] mid_CD2u column exists
[OK  ] mid_CD2l column exists
[OK  ] mid_CLCD0 column exists
[OK  ] mid_REref column exists
[OK  ] mid_REexp co